In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

root = Path("__file__").resolve().parents[1]
os.chdir(root)
pd.set_option("display.max_columns",None)

In [2]:
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

def collect(state: str = "FL", per_page: int = 100) -> pd.DataFrame:
    fields = ",".join([
        "school.name",

        # Prefer "latest" consistently
        "latest.programs.cip_4_digit.code",
        "latest.programs.cip_4_digit.title",
        "latest.programs.cip_4_digit.school.type",
        "latest.programs.cip_4_digit.credential.level",
        "latest.programs.cip_4_digit.distance",

        # Locale is school-level (not under programs)
        "latest.school.locale",

        # Might be valid; if you still get 500, try removing this one first
        "latest.school.carnegie_size_setting",

        # Fix typo: admission -> admissions
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type"
    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),
    }

    dfs = []

    # First request to get metadata/total pages
    params["page"] = "0"
    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    META = [
        "school.name",
        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type"
    ]

    # Helper to turn a page into a dataframe
    def page_to_df(data):
        results = [r for r in data.get("results", []) if r.get("latest.programs.cip_4_digit")]
        return pd.json_normalize(
        results,
        record_path=["latest.programs.cip_4_digit"],
        meta=META,
        errors="ignore",
        )

    dfs.append(page_to_df(data))

    # Fetch remaining pages (start at 1 to avoid refetching page 0)
    for page in range(1, total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        dfs.append(page_to_df(data))

    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    print(f"Total pages fetched: {total_pages}")
    print(f"Total Rows/Programs ingested: {len(df)}")
    return df

def get_json_or_raise(response: requests.Response):
    # Raise for HTTP errors early (4xx/5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    # Check content-type sanity (helps catch HTML responses)
    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    # Parse JSON with a clearer error if it fails
    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e
    
    import time
import random
import requests

def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
        if r.status_code < 500:
            return r
        last = r
        time.sleep((2 ** i) + random.random())
    return last

# Removed features
---
### 1

```
"latest.student.demographics.avg_family_income"
"latest.student.demographics.median_hh_income"
```
overlaps with ```"latest.student.demographics.median_family_income"```

---
### 2

```
"latest.academics.program_reporter.programs_offered"
```
~83% is null

---
### 3

```
latest.admissions.test_requirements
```
~46% is null and overlaps with ```latest.admissions.admission_rate.overall  ```

---
### 4

```
"latest.admissions.sat_scores.50th_percentile.critical_reading",
"latest.admissions.sat_scores.50th_percentile.math",
"latest.admissions.act_scores.50th_percentile.cumulative",
"latest.admissions.act_scores.50th_percentile.english",
"latest.admissions.act_scores.50th_percentile.math",
"latest.admissions.sat_scores.average.overall",
"latest.admissions.act_scores.midpoint.cumulative"
```
~58% is null and overlaps with ```latest.admissions.admission_rate.overall``` and ```latest.school.open_admissions_policy```



In [3]:
tdf = collect()
display(tdf.head())
display(tdf.info())

Total pages fetched: 4
Total Rows/Programs ingested: 8139


,code,title,distance,school.type,credential.level,school.name,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type
0,1101,"Computer and Information Sciences, General.",2,Public,1,Atlantic Technical College,21,-2,None,16748,None,1,26,1
1,1102,Computer Programming.,2,Public,1,Atlantic Technical College,21,-2,None,16748,None,1,26,1
2,1108,Computer Software and Media Applications.,2,Public,1,Atlantic Technical College,21,-2,None,16748,None,1,26,1
3,1109,Computer Systems Networking and Telecommunicat...,2,Public,1,Atlantic Technical College,21,-2,None,16748,None,1,26,1
4,1205,Culinary Arts and Related Services.,1,Public,1,Atlantic Technical College,21,-2,None,16748,None,1,26,1


<class 'pandas.DataFrame'>
RangeIndex: 8139 entries, 0 to 8138
Data columns (total 14 columns):
 #   Column                                            Non-Null Count  Dtype 
---  ------                                            --------------  ----- 
 0   code                                              8139 non-null   str   
 1   title                                             8139 non-null   str   
 2   distance                                          8139 non-null   int64 
 3   school.type                                       8139 non-null   str   
 4   credential.level                                  8139 non-null   int64 
 5   school.name                                       8139 non-null   str   
 6   latest.school.locale                              8139 non-null   object
 7   latest.school.carnegie_size_setting               8139 non-null   object
 8   latest.admissions.admission_rate.overall          4367 non-null   object
 9   latest.student.demographics.median_family

None

# Why so many missing Admission Rates?

In [4]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.4634)

In [5]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.0007)

Approximately 46% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [6]:
clean(df)

Numeric columns: Index(['distance', 'credential_level', 'admission_rate_overall',
       'median_family_income', 'students_with_pell_grant'],
      dtype='str')


,code,title,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type
0,1101,"Computer and Information Sciences, General.",2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
1,1102,Computer Programming.,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
2,1108,Computer Software and Media Applications.,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
3,1109,Computer Systems Networking and Telecommunicat...,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
4,1205,Culinary Arts and Related Services.,1,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8134,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",2,Herzing University-Tampa,21,-2,0.8696,18321.0,0.820345,2,31,1
8135,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",3,Herzing University-Tampa,21,-2,0.8696,18321.0,0.820345,2,31,1
8136,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",5,Herzing University-Tampa,21,-2,0.8696,18321.0,0.820345,2,31,1
8137,5138,"Registered Nursing, Nursing Administration, Nu...",1,"Private, for-profit",2,Galen Health Institutes-Miami Campus,21,-2,<NA>,32673.0,0.662796,1,30,1


In [7]:
display(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 8139 entries, 0 to 8138
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   code                       8139 non-null   string 
 1   title                      8139 non-null   string 
 2   distance                   8139 non-null   int64  
 3   school_type                8139 non-null   string 
 4   credential_level           8139 non-null   int64  
 5   school_name                8139 non-null   string 
 6   locale                     8139 non-null   string 
 7   carnegie_size_setting      8139 non-null   string 
 8   admission_rate_overall     4367 non-null   Float64
 9   median_family_income       8024 non-null   float64
 10  students_with_pell_grant   6957 non-null   Float64
 11  open_admissions_policy     8125 non-null   string 
 12  age_entry                  8024 non-null   string 
 13  title_iv_eligibility_type  8139 non-null   string 
dtypes: 

None

,code,title,distance,school_type,credential_level,school_name,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type
0,1101,"Computer and Information Sciences, General.",2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
1,1102,Computer Programming.,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
2,1108,Computer Software and Media Applications.,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
3,1109,Computer Systems Networking and Telecommunicat...,2,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1
4,1205,Culinary Arts and Related Services.,1,Public,1,Atlantic Technical College,21,-2,<NA>,16748.0,<NA>,1,26,1


In [8]:
tdf.tail()

,code,title,distance,school.type,credential.level,school.name,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type
8134,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",2,Herzing University-Tampa,21,-2,0.8696,18321,0.820345,2,31,1
8135,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",3,Herzing University-Tampa,21,-2,0.8696,18321,0.820345,2,31,1
8136,5202,"Business Administration, Management and Operat...",2,"Private, nonprofit",5,Herzing University-Tampa,21,-2,0.8696,18321,0.820345,2,31,1
8137,5138,"Registered Nursing, Nursing Administration, Nu...",1,"Private, for-profit",2,Galen Health Institutes-Miami Campus,21,-2,None,32673,0.662796,1,30,1
8138,1204,Cosmetology and Related Personal Grooming Serv...,1,"Private, for-profit",1,Salon Professional Academy-Elevate Salon Insti...,13,-2,None,13607,0.857143,1,26,1


In [9]:
save(df,"clean","ml_scorecard_FL_programs.csv")
save(tdf,file_type="",file_name="ml_scorecard_FL_programs.csv")